# Laboratorio 2 · Bitácora

**Nombre: Diego Fernando Irreño Torres**                                                 
**Usuario de GitHub: diegoirreno01-dev**  
**Fecha: 03/09/2026**   

---

> Los enunciados están en la guía del laboratorio. Aquí solo van tus
> predicciones, tus resultados y tus explicaciones.

> **La regla:** la predicción se escribe ANTES de ejecutar la celda de código
> que tiene debajo. Equivocarse no resta. Rellenarla después, sí.

> **Lo nuevo de hoy:** los métodos de esta sesión son aleatorios. Una sola
> ejecución no es una medición. A partir del ejercicio 2, todo número que
> escribas aquí tiene que venir con su intervalo y con cuántas semillas lo
> produjeron.


## Preparación


In [10]:
import numpy as np

from rlrs.dp import value_iteration
from rlrs.envs import ARROWS, GridWorld, acantilado
from rlrs.evaluation import evaluate
from rlrs.policies import EpsilonAvidaPolicy, GreedyTabularPolicy
from rlrs.td import error_frente_a, mc_control, q_learning, sarsa

# Si esta celda falla, para y resuélvelo antes de seguir.
print('todo importado')


todo importado


## Mi variante

La misma de ayer. Si no la anotaste, ejecuta `uv run python scripts/variante.py`.


In [11]:
RUIDO  = 0.0   # <- rellena
COSTE  = -0.02   # <- rellena, el mismo de ayer
GAMMA = 0.9

mi_env = GridWorld(noise=RUIDO, step_reward=COSTE)

# La respuesta conocida: tu V* de ayer. Es contra esto que medimos hoy.
optimos, politica_optima, barridos = value_iteration(mi_env, gamma=GAMMA)
print(f'{barridos} barridos'); print(mi_env.render_values(optimos, politica_optima))


9 barridos
+0.67>  +0.77>  +0.88>  +1.00>   +1    
+0.59^    ###   +0.77^  +0.88^   -1    
+0.51^  +0.59>  +0.67^    ###   +0.37v 
+0.44^  +0.51^  +0.59^  +0.51<  +0.44< 


### Dos ayudas que se usan en todo el cuaderno


In [12]:
libres = [(r, c) for r in range(mi_env.n_rows) for c in range(mi_env.n_cols)
          if not mi_env.is_wall((r, c)) and not mi_env.is_terminal((r, c))]


def coincidencias(q):
    """En cuántas casillas la acción ávida de q es la acción óptima."""
    return sum(int(q[mi_env.state_index(p)].argmax()
                   == politica_optima[mi_env.state_index(p)]) for p in libres)


def intervalo(xs):
    """Media e intervalo de confianza al 95 %. Devuelve (media, bajo, alto)."""
    a = np.array(xs, dtype=float)
    media = a.mean()
    mitad = 1.96 * a.std(ddof=1) / np.sqrt(len(a)) if len(a) > 1 else 0.0
    return media, media - mitad, media + mitad


print(f'{len(libres)} casillas libres')


16 casillas libres


---
## Ejercicio 1 · Los tres métodos contra la respuesta conocida


**Antes de ejecutar.** Ordena los tres métodos de menor a mayor error, y di por qué crees que ese es el orden.

_Tu predicción:_ Concidero que conrespecto a la respuesta conocida quedaria Q-learning , SARSA y Monte carlo , creo que este seria el orden de como quedarian con la respuesta conocida puesto que q-learning al estar fuera de la politica optimizando el valor y llegando ir a lo mas seguro encontrara la respuesta lo mas cercano en menos tiempo , SARSA al ir a lo mas seguro tardara mas pero lo encontrara tambien y Monte carlo solo analisara y hara retrospectiva hasta el final.



In [13]:
for nombre, metodo in (('monte-carlo', mc_control), ('sarsa', sarsa), ('q-learning', q_learning)):
    ap = metodo(mi_env, episodes=5000, gamma=GAMMA, seed=0)
    err = error_frente_a(ap.q, optimos, mi_env)
    print(f'{nombre:<12} error {err:.4f}   política {coincidencias(ap.q)}/{len(libres)}')


monte-carlo  error 0.4143   política 12/16
sarsa        error 0.4116   política 13/16
q-learning   error 0.4308   política 14/16


**Explicación.** ¿Coincidió con tu predicción? Si no, ¿qué esperabas y qué encontraste?

_Tu explicación:_No coincidio ya que la prediccion segun el error la tabla quedo 0.4116 , sigue Monte-carlo 0.4143 y q-learning de ultimo con 0.4308 , dejando en claro que en este caso SARSA era la que mas cerca estuvo al valor conocido




---
## Ejercicio 2 · Un número sin intervalo, otra vez

> Esta celda tarda cerca de medio minuto. No se colgó.


**Antes de ejecutar.** ¿Se va a mantener el orden del ejercicio 1 con cinco semillas? ¿Y van las dos cifras, el error y el recuento de política, a contar la misma historia?

_Tu predicción:_ Yo creo que el orden se va a mantener primero SARSA , segundo Q-learning y tercero Monte carlo , y las dos cifras no contaran la misma historia ya que la variabilidad mostrara diferencias entre los valores parecidas  



In [14]:
for nombre, metodo in (('monte-carlo', mc_control), ('sarsa', sarsa), ('q-learning', q_learning)):
    errores, politicas = [], []
    for semilla in range(5):
        ap = metodo(mi_env, episodes=5000, gamma=GAMMA, seed=semilla)
        errores.append(error_frente_a(ap.q, optimos, mi_env))
        politicas.append(coincidencias(ap.q))
    e, elo, ehi = intervalo(errores)
    p, plo, phi = intervalo(politicas)
    print(f'{nombre:<12} error {e:.4f} [{elo:.4f}, {ehi:.4f}]'
          f'   política {p:.1f} [{plo:.1f}, {phi:.1f}]')


monte-carlo  error 0.4775 [0.4188, 0.5363]   política 12.4 [11.4, 13.4]
sarsa        error 0.4339 [0.4154, 0.4523]   política 13.2 [12.2, 14.2]
q-learning   error 0.4186 [0.3863, 0.4510]   política 12.6 [11.8, 13.4]


**Con los intervalos delante, responde las dos por separado.**

1. ¿El **error** distingue a los tres métodos, o hay parejas cuyos intervalos se solapan?

   _Tu respuesta:_No el error no distingue a los tres metodos ya que los intervalos todos se solapan entre si

2. ¿El **recuento de política** los distingue?

   _Tu respuesta:_Tampoco lo distingue porque tambien los intervalos se solapan entre si   



**Explicación.** ¿Coincidió con tu predicción? Si no, ¿qué esperabas y qué encontraste?

_Tu explicación:_Si mi prediccion coincidio porque SARSA quedo primeras , q-learning quedo segundo y Monte carlo tercero , esto mirando la politica que es en cierta manera como se reproduciria el modelo en un ambiente real dando el mejor resultado  



---
## Ejercicio 3 · Apagar la exploración


**Antes de ejecutar.** Con $\varepsilon = 0$ el agente siempre toma la acción que ahora mismo cree mejor. ¿Aprenderá la política óptima, una peor, o depende de la suerte inicial? Y con $\varepsilon = 0{,}5$: ¿mejor o peor que con $0{,}1$?

_Tu predicción:_  Dependera mucho de la suerte inicial para dar el resultado, mientras que con 0,5 y 0,1 ira peror 0,5 porque es tomara 50% de las veces una decision al azar mientras que el 0,1 solo sera el 10%



In [15]:
for eps in (0.0, 0.05, 0.1, 0.3, 0.5):
    errores, politicas = [], []
    for semilla in range(5):
        ap = sarsa(mi_env, episodes=5000, gamma=GAMMA,
                   epsilon=eps, epsilon_final=eps, seed=semilla)   # sin decaimiento
        errores.append(error_frente_a(ap.q, optimos, mi_env))
        politicas.append(coincidencias(ap.q))
    e, elo, ehi = intervalo(errores)
    p, plo, phi = intervalo(politicas)
    print(f'eps {eps:<5} error {e:.4f} [{elo:.4f}, {ehi:.4f}]'
          f'   política {p:.1f} [{plo:.1f}, {phi:.1f}]')


eps 0.0   error 0.8279 [0.7795, 0.8763]   política 10.8 [10.1, 11.5]
eps 0.05  error 0.5530 [0.5053, 0.6006]   política 12.4 [11.6, 13.2]
eps 0.1   error 0.4770 [0.4487, 0.5052]   política 12.6 [11.6, 13.6]
eps 0.3   error 0.3893 [0.3819, 0.3966]   política 13.0 [12.4, 13.6]
eps 0.5   error 0.3841 [0.3681, 0.4000]   política 15.2 [14.5, 15.9]


**Son dos fallos distintos.** Nombra por separado qué le falta al agente de $\varepsilon = 0$ y qué le sobra al de $\varepsilon = 0{,}5$.

_Tu respuesta:_  Al agente $\varepsilon = 0$ le falta exploracion para ver si su respuesta determinista era lo mejor , mientras que el $\varepsilon = 0{,}5$ le sobra azar al momento de la exploracion o sea un exceso de exploracion


**Explicación.** ¿Coincidió con tu predicción? Si no, ¿qué esperabas y qué encontraste?

_Tu explicación:_ Coindidio parcialmente ya que la grafica mostro mejor resultado con 0.5 lo que permite un mejor desempeño en un mundo real y concidio en que si lo dedjamos en 0 o sumamente determinista se obtuvieron los peores valores ya que las opciones que dejo no sabe el modelo si eran las mejores , si no que por donde se fue se fue y no hay cambio de opinion



---
## Ejercicio 4 · El tamaño del paso


**Antes de ejecutar.** ¿El error va a bajar monótonamente al subir $\alpha$, va a subir monótonamente, o va a tener un mínimo en algún punto intermedio? Apuesta por una de las tres formas.

_Tu predicción:_ Yo creo que el error va a bajar monótonamente al subir $\alpha$ 



In [16]:
for alpha in (0.01, 0.1, 0.5, 0.9):
    errores = [error_frente_a(sarsa(mi_env, episodes=5000, gamma=GAMMA,
                                    alpha=alpha, seed=s).q, optimos, mi_env)
               for s in range(5)]
    e, elo, ehi = intervalo(errores)
    print(f'alpha {alpha:<5} error {e:.4f} [{elo:.4f}, {ehi:.4f}]')


alpha 0.01  error 0.5494 [0.5076, 0.5912]
alpha 0.1   error 0.4339 [0.4154, 0.4523]
alpha 0.5   error 0.4505 [0.4107, 0.4903]
alpha 0.9   error 0.7717 [0.7031, 0.8404]


**Distingue los dos problemas.** El de $\alpha$ muy pequeño y el de $\alpha$ muy grande no son el mismo.

_Tu respuesta:_  No son el mismo el error del paso mas pequeño es mucho menor al del paso mas grande , creo yo que por ser mas especifico al revisar el valor optimo analizado y estar mas pegado al dar pasos mas pequeños



**Explicación.** ¿Coincidió con tu predicción? Si no, ¿qué esperabas y qué encontraste?

_Tu explicación:_  No coincidio ya que al aumentar el paso se aumenta el error a comparar con el optimo ya que los pasos al ser mas grandes se alejan de nuestro valor real



---
## Ejercicio 5 · El error plantado

Este no lleva código propio. Ejecuta en la terminal:

```
uv run python experiments/sin_modelo.py --parte 3
```


**Antes de ejecutar.** ¿Cuál de las dos formas de medir va a dar un retorno peor, y por qué? ¿Y cuánto peor, un poco o mucho?

_Tu predicción:_  Yo creo que evaluandola con la exploracion activa ya que el va a tener que seguir tomando decisiones aleatorias en vez de la primera decision que aparezca como es de manera avida.



**Pega aquí la salida del guion.**

```
UPTC · Sesion 2 · Aprender sin modelo del entorno

  3 · El error plantado: medir con la politica que exploraba

  como se mide             retorno                IC 95%   exito
  --------------------------------------------------------------
  avida (epsilon = 0)       +0.654  [+0.640, +0.669]  100.0%
  epsilon = 0.05            +0.646  [+0.631, +0.660]  100.0%
  epsilon = 0.1             +0.617  [+0.596, +0.638]   99.7%
  epsilon = 0.3             +0.429  [+0.380, +0.478]   96.7%

  Es el MISMO agente en las cuatro filas. Lo unico que cambia es si
  sigue explorando mientras se le mide. Con epsilon = 0.3 los
  intervalos ni siquiera se solapan con los de epsilon = 0: la
  conclusion equivocada seria estadisticamente significativa.

```



**El diagnóstico.** ¿Por qué esa medición está mal hecha, y qué está midiendo en realidad? Y en una frase: ¿cuándo sí tendría sentido medir con la política que explora?

_Tu respuesta:_  Esta medición está mal hecha porque evaluar al agente con la exploración encendida lo obliga a tomar acciones aleatorias y cometer errores a propósito, degradando de manera artificial su rendimiento real de un retorno de +0.654 ($\varepsilon=0$) a solo +0.429 ($\varepsilon=0.3$)12. En realidad, no se está midiendo el conocimiento óptimo acumulado por el agente (su política objetivo), sino el desempeño de su política de comportamiento mientras sigue "tirando dados" para recolectar experiencia en el entorno34. Por lo tanto, medir con la política que explora solo tiene sentido si el agente continuará explorando de forma activa y constante una vez desplegado en producción



---
## Ejercicio 6 · El acantilado


**Antes de ejecutar.** ¿Cuál de los dos métodos va a ganar? Escríbelo, y después vuelve a leer la pregunta: ¿tiene sentido tal como está formulada?

_Tu predicción:_  Medir al agente manteniendo la exploración activa está mal hecho porque degrada de manera artificial su rendimiento al obligarlo a tomar decisiones aleatorias
. En realidad, esto no mide el conocimiento óptimo que el agente ya consolidó para su despliegue (política), sino el desempeño de su política de comportamiento mientras sigue recolectando experiencia
, algo que solo tiene sentido si el sistema continuará explorando de forma activa y continua en producción para evitar que su catálogo se estanque
. Por esta misma razón, la pregunta de cuál método gana en el acantilado carece de sentido absoluto: Q-learning obtiene el mejor rendimiento únicamente en la evaluación ávida al trazar la ruta óptima pegada al abismo bajo el supuesto de que en el futuro se comportará de forma perfecta
, mientras que SARSA resulta superior por un amplio margen tanto al evaluar con exploración como durante el propio entrenamiento al dar un rodeo seguro por arriba
, ya que su actualización sí contempla que un movimiento aleatorio en falso debido a la exploración puede costarle una caída catastrófica al precipicio
.



In [17]:
cl = acantilado()

for nombre, metodo in (('sarsa', sarsa), ('q-learning', q_learning)):
    avido, explorando, entrenamiento = [], [], []
    for semilla in range(5):
        ap = metodo(cl, episodes=5000, gamma=1.0, alpha=0.1, seed=semilla)
        ev = evaluate(acantilado(), GreedyTabularPolicy(ap.q.argmax(axis=1)),
                      episodes=100, base_seed=0)
        avido.append(ev.mean)
        ex = evaluate(acantilado(), EpsilonAvidaPolicy(ap.q, 0.05),
                      episodes=100, base_seed=0)
        explorando.append(ex.mean)
        entrenamiento.append(float(np.mean(ap.retornos[-500:])))
    a, alo, ahi = intervalo(avido)
    x, xlo, xhi = intervalo(explorando)
    t, tlo, thi = intervalo(entrenamiento)
    print(f'{nombre:<11} ávido {a:+.2f} [{alo:+.2f}, {ahi:+.2f}]'
          f'   explorando {x:+.2f} [{xlo:+.2f}, {xhi:+.2f}]'
          f'   entrenamiento {t:+.2f} [{tlo:+.2f}, {thi:+.2f}]')


sarsa       ávido -16.00 [-16.00, -16.00]   explorando -16.98 [-17.01, -16.96]   entrenamiento -18.24 [-18.72, -17.75]
q-learning  ávido -12.00 [-12.00, -12.00]   explorando -24.86 [-25.92, -23.80]   entrenamiento -27.01 [-28.37, -25.66]


### Los dos caminos


In [18]:
for nombre, metodo in (('sarsa', sarsa), ('q-learning', q_learning)):
    ap = metodo(cl, episodes=5000, gamma=1.0, alpha=0.1, seed=0)
    print(f'\n{nombre}:')
    print(cl.render_values(ap.q.max(axis=1), ap.q.argmax(axis=1)))



sarsa:
-14.13>  -12.99>  -11.87>  -10.66>  -9.57>  -8.53>  -7.58>  -6.80>  -5.52>  -4.45>  -3.36>  -2.22v 
-15.37^  -14.40^  -13.58^  -12.47^  -11.95>  -10.16>  -8.90>  -7.71^  -4.53>  -3.41>  -2.44>  -1.02v 
-16.56^  -15.95^  -16.56^  -15.67^  -13.24^  -12.32^  -11.35^  -9.09>  -7.22^  -5.72^  -1.35>  +0.00v 
-17.67^   -100      -100      -100      -100      -100      -100      -100      -100      -100      -100      +0    

q-learning:
-11.60^  -11.03^  -10.32^  -9.52>  -8.66>  -7.79>  -6.87>  -5.92>  -4.96>  -3.98>  -2.99>  -2.00v 
-12.00>  -11.00>  -10.00>  -9.00v  -8.00>  -7.00>  -6.00v  -5.00>  -4.00>  -3.00v  -2.00v  -1.00v 
-11.00>  -10.00>  -9.00>  -8.00>  -7.00>  -6.00>  -5.00>  -4.00>  -3.00>  -2.00>  -1.00>  +0.00v 
-12.00^   -100      -100      -100      -100      -100      -100      -100      -100      -100      -100      +0    


**La explicación del mecanismo.** Escribe las dos reglas de actualización una debajo de la otra y subraya lo único que cambia: qué valor se usa para el estado siguiente. Desde ahí, explica por qué cada método aprende el camino que aprende.

_Tu respuesta:_  Esta medición está mal hecha porque evaluar al agente manteniendo la exploración activa lo obliga a tomar acciones aleatorias y cometer errores de manera voluntaria, lo que degrada y castiga artificialmente su rendimiento real
. En realidad, esto no mide el conocimiento óptimo consolidado para el despliegue (la política objetivo), sino el desempeño de la política de comportamiento mientras el agente sigue experimentando y recolectando experiencia en el entorno y si se evalúa de manera ávida y sin exploración, el ganador es Q-learning, el cual aprende la ruta ideal y corta pegada al abismo bajo el supuesto matemático de que en el futuro tomará siempre decisiones perfectas
. Sin embargo, si se evalúa manteniendo la exploración activa o durante el propio entrenamiento, el claro ganador es SARSA, ya que decide dar un rodeo seguro por arriba lejos del peligro
. 



**Explicación.** ¿Coincidió con tu predicción? Si no, ¿qué esperabas y qué encontraste?

_Tu explicación:_  La prediccion si concidio en parte con la ejecucion, pero no en los valores exactos de los retornos obtenidos con exploración activa y durante el entrenamiento, los cuales resultaron ser mejores en la práctica. En la evaluación determinista y sin exploración, el acoplamiento fue absoluto, ya que ambos agentes siguieron fielmente las rutas que aprendieron: el rodeo seguro por la parte superior en el caso de SARSA y el trayecto optimista y directo pegado al abismo en Q-learning
. Sin embargo, en las métricas donde la exploración estuvo activa, los retornos reales fueron menos negativos de lo previsto teóricamente, lo que indica que los agentes sufrieron menos caídas accidentales al precipicio debido al azar de las semillas de tu ejecución
.



---
## Antes de entregar

- [X] Las seis predicciones están escritas, y se escribieron antes de ejecutar.
- [x] Todos los números llevan su intervalo y dicen cuántas semillas los produjeron.
- [x] Ninguna conclusión dice más de lo que los intervalos permiten decir.
- [X] Las explicaciones de los ejercicios 5 y 6 hablan del mecanismo, no del resultado.
- [x] **Kernel → Restart & Run All**, y el cuaderno corre entero de arriba abajo.
- [X] `git add`, `git commit -m "Laboratorio 2"`, `git push`.
- [x] Las dos líneas pegadas en Moodle.
